# unsloth
- unsloth ? 최근 LLM 파인튜닝의 최적화 분야에서 많은 사용자들의 지지를 받는 솔루션
- 최적화 ? 파인튜닝 시 필요한 메모리 사용을 줄이고, 학습 속도를 높이는 것
- 강점
  - 학습 속도의 향상
  - 메모리 사용량 감소
  - 다양한 하드웨어 및 허깅페이스 생태계와의 호환성

- install unsloth on computer   
(https://github.com/unslothai/unsloth#installation-instructions---conda).


- This notebook uses the `Llama-3` format for conversation style finetunes. We use [Open Assistant conversations](https://huggingface.co/datasets/philschmid/guanaco-sharegpt-style) in ShareGPT style.

In [1]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

* We support Llama, Mistral, Phi-3, Gemma, Yi, DeepSeek, Qwen, TinyLlama, Vicuna, Open Hermes etc
* We support 16bit LoRA or 4bit QLoRA. Both 2x faster.
* `max_seq_length` can be set to anything, since we do automatic RoPE Scaling via [kaiokendev's](https://kaiokendev.github.io/til) method.
* With [PR 26037](https://github.com/huggingface/transformers/pull/26037), we support downloading 4bit models **4x faster**! [Our repo](https://huggingface.co/unsloth) has Llama, Mistral 4bit models.
* [**NEW**] We make Phi-3 Medium / Mini **2x faster**! See our [Phi-3 Medium notebook](https://colab.research.google.com/drive/1hhdhBa1j_hsymiW9m-WzxQtgqTH_NHqi?usp=sharing)

In [2]:
from unsloth import FastLanguageModel #unsloth에서 가장 중요한 클래스(빠른 언어 모델 클래스)
import torch
max_seq_length = 2048 # 최대 시퀀스 길이 설정 (너무 길면 VRAM 요구량 증가)
dtype = None # 자동 감지를 위해 none 사용
load_in_4bit = True # 4BIT 양자화 사용

# 4bit 사전 양자화 모델
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Choose ANY!
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.1: Fast Llama patching. Transformers: 4.52.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

## PEFT 적용을 위한 LoRA(low rank adaptation) 어댑터 추가 -> Fine tuning의 효율성 증가
- 일부 layer들에 대해서 weight를 선택하여 학습

In [3]:
# 모델 초기화
# PEFT 적용을 위한 LoRA 어댑터 추가
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 0보다 큰 어떤 숫자도 선택 가능. 8, 16, 32, 64, 128 권장
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  #LORA 알파값 설정
    lora_dropout = 0, #드롭아웃 지원 (과적합 줄이기)
    bias = "none",    # bias 지원
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # 순위 안정화 lora 지원
    loftq_config = None, # LoftQ 지원
)

Unsloth 2025.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep
- llama3 포멧 사용 (대화 형식의 fine tuning)
- 예제 : 사람-gpt 형식의 대화 로그 기록  [Open Assistant conversations](https://huggingface.co/datasets/philschmid/guanaco-sharegpt-style) in ShareGPT style. Llama-3 renders multi turn conversations(멀티 턴 chat format) like below:  

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

- |begin_of_text| : 대화 시퀀스의 시작임을 알려줌
- |start_header_id|>user<|end_header_id| : user의 발화가 나올 차례
- |start_header_id|>assistant<|end_header_id| : 어시스턴트(모델)이 발화할 차례
- |eot_id| : 발화 마무리

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old` and our own optimized `unsloth` template.

Note ShareGPT uses `{"from": "human", "value" : "Hi"}` and not `{"role": "user", "content" : "Hi"}`, so we use `mapping` to map it.

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [4]:
#Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Fine tuining 을 위해 준비한 JSONL 데이터셋 로드: columns = ["text"]
from datasets import load_dataset
chat_jsonl_path = "/content/drive/MyDrive/digital_smart_busan/DSBA_project_2/데이터셋/pan12_llama3_chat_format_manual.jsonl"
dataset = load_dataset("json", data_files={"train": chat_jsonl_path}, split="train")

# 데이터셋 속성 확인
print("Features:", dataset.features)
print("샘플 text:", dataset[0]["text"][:200], "...")

Generating train split: 0 examples [00:00, ? examples/s]

Features: {'text': Value(dtype='string', id=None)}
샘플 text: <|begin_of_text|><|start_header_id|>user<|end_header_id|>

hey!!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

hi! how are you feeling?<|eot_id|> ...


- tokenization
  - 문자열을 모델이 처리할 수 있게 인덱스로 바꿔주기
  - 모델의 입력 포멧
    - 모든 LLM은 '입력 -> 숫자 토큰 시퀀스(token IDs) -> 임베딩(벡터) -> 신경망' 과정을 거치기 때문에 토크나이제이션 필요
    

- dataset.map
  - dataset 라이브러리(hugging face datasets)는 대용량 데이터셋을 편하게 로드하고 조작할 수 있도록 돕는 도구
  - dataset = load_dataset('json', data_files=...,split="train) 로 json 파일을 불러오면 각 예시가 하나의 딕셔너리로 되어있는 datasets.Dataset 객체로 만들어짐
    - {"text": "<|begin_of_text|>…<|eot_id|>"}

In [7]:
def tokenize_fn(examples):
    # examples["text"]는 ["<|begin_of_text|>…", "<|begin_of_text|>…", …]
    encodings = tokenizer(
        examples["text"],
        padding="longest",
        truncation=True,
        max_length=max_seq_length
    )
    encodings["labels"] = encodings["input_ids"].copy()
    return encodings

# map을 통해 토크나이징 적용 → text 컬럼 제거
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)

# 적용 결과 확인
print("토크나이즈 후 샘플:", tokenized[0])


Map:   0%|          | 0/68489 [00:00<?, ? examples/s]

토크나이즈 후 샘플: {'input_ids': [128000, 128000, 128006, 882, 128007, 271, 36661, 3001, 128009, 128006, 78191, 128007, 271, 6151, 0, 1268, 527, 499, 8430, 30, 128009, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255, 128255], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

- 래핑
  - 현재 jsonl 데이터는 여러개의 리스트가 모여 있는 형태
  - input_ids = [[],,,,[]], attention_mask =[[]...[]], labels=[[]...[]]
  - 즉, 한 샘플마다 토큰 id, 어텐션 마스크를 별도로 리스트로 모아둔 상태
  - pytorch 를 통해 한번에 한 샘플이 아닌 , 한번에 여러 샘플을 꺼내 모델 학습
  - {
  - "input_ids": tensor([...]),
  - "attention_mask": tensor([...]),
  - "labels": tensor([...])
  - }
  - 형태의 딕셔너리로 반환

In [8]:
import torch
from torch.utils.data import Dataset

class ChatDataset(Dataset):
    def __init__(self, hf_dataset):  #생성자 (어떤 데이터를 다룰지 정해준다)
        # hf_dataset: Hugging Face Dataset 객체 (토크나이징 후 결과)
        self.dataset = hf_dataset
        # Hugging Face Dataset의 컬럼 이름 목록을 미리 저장
        self.columns = hf_dataset.column_names

    def __len__(self):
        # 전체 샘플 개수
        return len(self.dataset)

    def __getitem__(self, idx):
        # idx번째 샘플을 꺼내서, torch.tensor로 만들어 반환
        item = {}
        for col in self.columns:
            # 각 컬럼별로 "hf_dataset[col][idx]"가 int 리스트(또는 이미 int) 형태
            item[col] = torch.tensor(self.dataset[col][idx])
        return item

# 사용 예시
train_dataset = ChatDataset(tokenized)

print("Dataset 크기:", len(train_dataset))

# 첫 번째 샘플을 item 변수에 저장
item0 = train_dataset[0]

# 각 컬럼별 tensor의 shape 만 출력
for col, tensor in item0.items():
    print(f"- {col}.shape: {tuple(tensor.shape)}")

Dataset 크기: 68489
- input_ids.shape: (95,)
- attention_mask.shape: (95,)
- labels.shape: (95,)


### 예제 data prep


In [4]:
from unsloth.chat_templates import get_chat_template
#토크나이저 래핑 : shareGPT 스타일 대화 -> LLAMA3 CHAT 포멧열 기능 추가
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3", # Supports zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)

def formatting_prompts_func(examples):  #HuggingFace Dataset에 있는 “conversations” 컬럼을
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass
#데이터셋 로드
from datasets import load_dataset
dataset = load_dataset("philschmid/guanaco-sharegpt-style", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,) #MAP을 돌려 conversations를 text 로 변환

README.md:   0%|          | 0.00/442 [00:00<?, ?B/s]

(…)-00000-of-00001-8aae24b47ddaaf21.parquet:   0%|          | 0.00/8.24M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9033 [00:00<?, ? examples/s]

Map:   0%|          | 0/9033 [00:00<?, ? examples/s]

Let's see how the `Llama-3` format works by printing the 5th element

In [5]:
dataset[5]["conversations"]

[{'from': 'human',
  'value': 'What is the typical wattage of bulb in a lightbox?'},
 {'from': 'gpt',
  'value': 'The typical wattage of a bulb in a lightbox is 60 watts, although domestic LED bulbs are normally much lower than 60 watts, as they produce the same or greater lumens for less wattage than alternatives. A 60-watt Equivalent LED bulb can be calculated using the 7:1 ratio, which divides 60 watts by 7 to get roughly 9 watts.'},
 {'from': 'human',
  'value': 'Rewrite your description of the typical wattage of a bulb in a lightbox to only include the key points in a list format.'}]

In [6]:
print(dataset[5]["text"])

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

What is the typical wattage of bulb in a lightbox?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The typical wattage of a bulb in a lightbox is 60 watts, although domestic LED bulbs are normally much lower than 60 watts, as they produce the same or greater lumens for less wattage than alternatives. A 60-watt Equivalent LED bulb can be calculated using the 7:1 ratio, which divides 60 watts by 7 to get roughly 9 watts.<|eot_id|><|start_header_id|>user<|end_header_id|>

Rewrite your description of the typical wattage of a bulb in a lightbox to only include the key points in a list format.<|eot_id|>


If you're looking to make your own chat template, that also is possible! You must use the Jinja templating regime. We provide our own stripped down version of the `Unsloth template` which we find to be more efficient, and leverages ChatML, Zephyr and Alpaca styles.

More info on chat templates on [our wiki page!](https://github.com/unslothai/unsloth/wiki#chat-templates)

In [ ]:
unsloth_template = \
    "{{ bos_token }}"\
    "{{ 'You are a helpful assistant to the user\n' }}"\
    "{% for message in messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ '>>> User: ' + message['content'] + '\n' }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ '>>> Assistant: ' + message['content'] + eos_token + '\n' }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}"\
        "{{ '>>> Assistant: ' }}"\
    "{% endif %}"
unsloth_eos_token = "eos_token"

if False:
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = (unsloth_template, unsloth_eos_token,), # You must provide a template and EOS token
        mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
        map_eos_token = True, # Maps <|im_end|> to </s> instead
    )

<a name="Train"></a>
### 모델 학습
- 로컬에서 실행할 경우, CUDA Toolkit 설치 필요
  - nvdia cuda toolkit 전체 설치 필요 (bash)
    - sudo apt-get update
    - sudo apt-get install -y nvidia-cuda-toolkit

Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel

#Trainer 설정 및 학습 실행 (argument : 함수, 메서드 입력값(value))
training_args = TrainingArguments(
    output_dir = "./llama3_chat_ft",       # 체크포인트 저장 디렉터리
    per_device_train_batch_size = 1,        # GPU 하나당 배치 크기 (LoRA+4bit라 1~2 정도 권장)
    gradient_accumulation_steps = 4,        # 배치 4개를 모아서 역전파(실제 배치 크기 = 1*4 = 4)
    num_train_epochs = 3,                   # 전체 에폭 수 (필요에 따라 조정)
    learning_rate = 3e-5,
    fp16 = True,                            # FP16 혼합 정밀도 (T4에서 가능)
    logging_steps = 50,                     # 50 스텝마다 로그 기록
    save_steps = 500,                       # 500 스텝마다 체크포인트 저장
    save_total_limit = 2,                   # 저장할 체크포인트 수 제한
    optim = "adamw_8bit",                   # 8-bit AdamW 옵티마이저
    weight_decay = 0.01,                    # 가중치 감쇠
    lr_scheduler_type = "linear",           # 선형 스케줄러
    seed = 3407,
    report_to = "none",                     # WandB/TensorBoard 비활성화
)

trainer = Trainer(
    model = model,                          # LoRA+4bit 양자화된 Llama-3 모델
    args = training_args,
    train_dataset = train_dataset,          # 래핑된 PyTorch Dataset
    tokenizer = tokenizer,                  # 같은 토크나이저를 넘겨주면 DataCollator 자동 사용
)
# 학습 실행 및 결과(TrainOutput) 저장
trainer_stats = trainer.train()

# 반환된 TrainOutput 확인
print(trainer_stats)


# 학습 완료 후 모델 저장
trainer.save_model("/content/drive/MyDrive/digital_smart_busan/DSBA_project_2/데이터셋/llama3_8b_chat_ft")
print("✅ Fine‐tuning 완료, 모델이 저장되었습니다.")

<ipython-input-9-bc1ab8fbfeaf>:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,489 | Num Epochs = 3 | Total steps = 51,369
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss


In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference (추론)
- Let's run the model! Since we're using `Llama-3`, use `apply_chat_template` with `add_generation_prompt` set to `True` for inference.
- 챗봇처럼 질문을 던지고 답변 생성 (원하는 페르소나로 대화 생성할 수 있는지 확인)  

In [ ]:
from unsloth.chat_templates import get_chat_template

#토크나이저를 다시 chat 포멧으로 래핑
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3", # Supports zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")


outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model") # Local saving
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "What is a famous tall tower in Paris?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoModelForPeftCausalLM
    from transformers import AutoTokenizer
    model = AutoModelForPeftCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in `llama.cpp` or a UI based system like `GPT4All`. You can install GPT4All by going [here](https://gpt4all.io/index.html).

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/u54VK8m8tk) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Zephyr DPO 2x faster [free Colab](https://colab.research.google.com/drive/15vttTpzzVXv_tJwEk-hIcQ0S9FcEWvwP?usp=sharing)
2. Llama 7b 2x faster [free Colab](https://colab.research.google.com/drive/1lBzz5KeZJKXjvivbYvmGarix9Ao6Wxe5?usp=sharing)
3. TinyLlama 4x faster full Alpaca 52K in 1 hour [free Colab](https://colab.research.google.com/drive/1AZghoNBQaMDgWJpi4RbffGM1h6raLUj9?usp=sharing)
4. CodeLlama 34b 2x faster [A100 on Colab](https://colab.research.google.com/drive/1y7A0AxE3y8gdj4AVkl2aZX47Xu3P1wJT?usp=sharing)
5. Mistral 7b [free Kaggle version](https://www.kaggle.com/code/danielhanchen/kaggle-mistral-7b-unsloth-notebook)
6. We also did a [blog](https://huggingface.co/blog/unsloth-trl) with 🤗 HuggingFace, and we're in the TRL [docs](https://huggingface.co/docs/trl/main/en/sft_trainer#accelerate-fine-tuning-2x-using-unsloth)!
7. Text completions like novel writing [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing)
9. Gemma 6 trillion tokens is 2.5x faster! [free Colab](https://colab.research.google.com/drive/10NbwlsRChbma1v55m8LAPYG15uQv6HLo?usp=sharing)

<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Support our work if you can! Thanks!
</div>